# TravelMind — Notebook rattrapage consolidé

Ce notebook remplace la version de travail dense par une version plus lisible, recentrée sur les améliorations de la version rattrapage.

Fil conducteur retenu :

- utiliser le dataset enrichi `v2.2-minimal` comme référence unique ;
- clarifier la finalité métier et la plus-value ;
- éviter les doublons entre anciennes expériences et version rattrapage ;
- converger vers le modèle final : `LogisticRegression_balanced_optimized` ;
- présenter uniquement les métriques utiles à la décision : `accuracy`, `balanced_accuracy`, `precision`, `recall`, `F1`, `ROC AUC`, matrice de confusion et diagnostic d'overfitting.


# 📘 Cas d'usage — Certification "Concevoir et implémenter une solution d'IA"


Nom Prénom : EL OUARDI Mina <br>
Formation : Concevoir une solution IA <br> 
Date : 26/06/2026


## Introduction


**Nom du projet : TravelMind** — solution IA destinée à aider une agence de voyages haut de gamme à anticiper le risque de satisfaction faible ou moyenne avant le départ.

Ce notebook consolidé suit une logique plus directe que la version exploratoire :

1. **Cadrage et datasheet** : besoin métier, KPI, contraintes, dataset et limites.
2. **Éthique, RGPD et AI Act** : risques, biais, supervision humaine et conformité.
3. **Recentrage rattrapage** : clarification de la finalité et de la plus-value.
4. **Préparation des données** : construction du dataset `v2.2-minimal`, contrôles, nettoyage et feature engineering.
5. **Modélisation finale** : classification binaire pré-voyage, équilibrage des classes, optimisation et diagnostic.
6. **Industrialisation** : architecture, API, monitoring, impacts et amélioration continue.

Les anciennes pistes exploratoires restent utiles comme historique, mais elles ne sont plus répétées ici afin de rendre le livrable plus lisible et convergent.


## 0. Journal de bord du projet TravelMind

Ce journal de bord synthétise les principales décisions prises pendant le projet. Il permet de suivre l'avancement, la raison de chaque action, les preuves associées et la décision retenue pour le livrable final.

| Étape | Action réalisée | Objectif | Résultat obtenu | Preuve / suite |
| --- | --- | --- | --- | --- |
| Initialisation | Création de l'environnement Python `.venv`, du dépôt Git/GitHub et de la base Docker | Rendre le projet reproductible, versionné et portable | Structure projet en place avec dépendances, dépôt Git et conteneurisation | `requirements.txt`, `.gitignore`, `Dockerfile`, `docker-compose.yml` |
| Cadrage métier | Définition du besoin TravelMind, des cas d'usage pré-voyage et post-voyage, des KPI et des contraintes | Clarifier le problème métier et vérifier que l'IA apporte une valeur | Deux objectifs distingués : prédiction avant départ et analyse qualité après séjour | KPI métier, séparation pré/post-voyage
| Dataset | Identification du fichier `Examen_travel_planning_dataset.csv` et vérification des droits d'usage | Vérifier la pertinence, la disponibilité et la cohérence du jeu de données | Dataset synthétique, anonymisé et utilisable dans le cadre de ce projet | `data/versions/v2_2_signal_enrichment/travel_planning_dataset_v2_2_minimal.csv`|
| Datasheet et gouvernance | Documentation de la motivation, composition, usages, limites, distribution et maintenance du dataset | Répondre aux exigences de documentation du jeu de données | Datasheet structurée ajoutée au notebook |
| Éthique, RGPD et AI Act | Analyse des risques, registre RGPD, rôles AI Act, usage clients finaux et obligation d'alphabétisation IA | Encadrer l'usage responsable de TravelMind | Risques, mesures, supervision humaine |
| Analyse métier du dataset brut | Contrôle de l'unicité, satisfaction, budget, prix vol, activités, imprévus, réorganisation et fuite de données | Identifier les cas incohérents ou explicables avant préparation | Incohérences critiques distinguées des observations métier explicables| tableaux de contrôles |
| Préparation des données | Nettoyage, valeurs manquantes, outliers, valeurs négatives, doublons et formats | Renforcer l'intégrité des données avant modélisation | Traitements documentés et appliqués | pipelines sklearn |
| Pipeline | Incluson de l'imputation, encodage, standardisation et traitement IQR dans les pipelines après split train/test | Éviter que le test influence les paramètres appris sur le train | Pipeline plus rigoureux et conforme aux bonnes pratiques ML | `ColumnTransformer`, `Pipeline`, `IQRMedianOutlierReplacer` |
| Feature engineering | Création de variables dérivées pré-voyage et post-voyage explicatives | Ajouter du signal métier tout en respectant le moment d'utilisation | Features pré-voyage retenues pour l'API, variables post-voyage exclues du modèle pré-voyage | Sections feature engineering et liste des features |
| Versioning des données | Mise en place des versions v1.0, v1.1, v1.2, v2.0 et v2.1 | Tracer les transformations du dataset et garantir la reproductibilité | Versions documentées et générées dans `data/versions/` | `scripts/version_data.py`, `docs/data_versioning.md` |
| Modélisation pré-voyage | Comparaison de modèles 3 classes avec baseline, validation croisée et diagnostics | Tester la capacité des variables avant départ à prédire la satisfaction | Performance modeste, cohérente avec le faible signal pré-voyage | Retenu comme modèle industrialise car il correspond au cas d'usage avant départ |
| Expériences pré-voyage | Tests Optuna, RandomizedSearch, SMOTE, SMOTENC, augmentation 20 000 et 50 000 lignes | Chercher un gain de performance sans changer l'objectif métier | Gains limités ou instables, parfois surapprentissage | pour industrialisation | Notebook de tests hyperparamètres, synthèse des essais |
| Modélisation post-voyage | Ajout des variables opérationnelles `imprevus`, `respect_budget`, `reorganisation_necessaire` pour analyse qualité | Comparer le signal disponible après séjour avec le signal pré-voyage | Scores meilleurs, confirmant que les variables post-voyage expliquent davantage la satisfaction | Conservé pour analyse qualité, non retenu pour l'API pré-voyage | Sections post-voyage |
| NLP exploratoire | Analyse de `retour_client` avec tokenisation, lemmatisation, POS, NER et sentiment | Évaluer l'apport du texte libre | NLP utile pour l'analyse qualitative, mais non retenu dans le modèle principal en raison du risque de fuite du ressenti client | Conservé comme exploration | Section NLP exploratoire |
| Évaluation et transparence | Ajout matrice de confusion, validation croisée, overfitting, underfitting, Model Card, métriques éthiques et CodeCarbon | Documenter la performance, les limites et les impacts | Lecture métier des scores, biais et empreinte carbone documentée | Model Card, CodeCarbon |
| Industrialisation | Création de `train.py`, `app/`, TravelMind API, monitoring, logs et TravelMind Dashboard | Servir le modèle pré-voyage hors notebook | API locale fonctionnelle avec `/health`, `/predict`, `/monitoring/*` et interface web | `train.py`, `app/main.py`, `app_web.py` |
| Monitoring et réentraînement | Journalisation JSONL, drift, alertes, dashboard KPI et stratégie de réentraînement | Suivre l'exploitation et déclencher des actions si le contexte change | Monitoring initial et seuils d'alerte disponibles | `app/monitoring.py`, `docs/strategie_reentrainement.md` |
| CI/CD et qualité | Ajout des tests, GitHub Actions | Automatiser les contrôles avant livraison | Tests API/ML/monitoring et contrôle qualité modèle intégrés | `.github/workflows/ci-cd.yml`, `tests/`, `configs/model_quality_gate.json` |
| Architecture cible | Architecture retenue : API locale + dashboard + Docker optionnel + CI/CD | Choisir une architecture proportionnée au prototype | Architecture retenue : API locale + dashboard + Docker optionnel + CI/CD |


## Partie 1 - Cadrage, besoins métier, cas d'usage et dataset

Cette partie fixe le cadre du projet TravelMind avant toute préparation de données ou modélisation. Elle précise le problème métier, les objectifs IA, les KPI, les contraintes et le rôle du dataset.

### 1.1 Définition du problème métier

L'agence de voyages haut de gamme souhaite mieux personnaliser ses propositions de séjour et anticiper le niveau de satisfaction client. Le problème concret est d'identifier, avant le départ, les séjours présentant un risque d'insatisfaction afin d'aider le conseiller à ajuster la proposition : budget, destination, durée, hébergement, météo prévue ou activité principale.

Le projet distingue deux objectifs :

| Objectif | Moment d'utilisation | Rôle dans le projet |
| --- | --- | --- |
| Pré-voyage | Avant le départ | Objectif principal industrialisé : prédire une classe de satisfaction à partir des informations disponibles avant le séjour. |
| Post-voyage | Pendant ou après le séjour | Objectif exploratoire : comprendre l'apport des imprévus, du respect du budget, de la réorganisation et des retours clients. |

### 1.2 Cas d'usage retenus

- **Aide à la décision pré-voyage** : signaler au conseiller les voyages potentiellement risqués avant validation finale.
- **Analyse qualité post-voyage** : comprendre les facteurs associés à la satisfaction ou à l'insatisfaction après séjour.
- **Amélioration continue** : exploiter les résultats, les logs, les retours clients et les nouvelles données pour améliorer progressivement le service.

Les variables post-voyage (`imprevus`, `reorganisation_necessaire`, `respect_budget`, `retour_client`) sont exclues du modèle pré-voyage afin d'éviter une fuite de données.

### 1.3 Indicateurs de succès du projet

| Type de KPI | Indicateur | Utilité |
| --- | --- | --- |
| Métier | Taux de satisfaction client | Mesurer la part des séjours satisfaisants dans le dataset et suivre l'amélioration future du service. |
| Métier | Taux d'insatisfaction client | Identifier la proportion de séjours à risque et prioriser les actions de revue humaine. |
| Technique | `macro_f1` | Mesurer la performance moyenne sur les classes sans favoriser la classe majoritaire. |
| Technique | `balanced_accuracy` | Vérifier que le modèle reste pertinent malgré un déséquilibre de classes. |
| Technique | Matrice de confusion | Comprendre les erreurs entre insatisfait, neutre et satisfait. |
| Technique | Validation croisée | Vérifier la stabilité du modèle sur plusieurs découpages de données. |

### 1.4 Contraintes du projet

- **Données** : dataset synthétique, anonymisé, de taille modérée ; il ne reflète pas toute la complexité d'une activité réelle.
- **Signal disponible** : les variables pré-voyage expliquent faiblement la satisfaction, ce qui limite mécaniquement la performance attendue.
- **Éthique et conformité** : le modèle doit rester un outil d'aide à la décision, avec supervision humaine et sans décision automatique défavorable pour le client.
- **Industrialisation** : le prototype doit être reproductible avec Git, Docker, scripts d'entraînement, API, interface web, tests et monitoring local.
- **Déploiement** : le déploiement distant automatique n'est pas activé dans cette version ; il reste une étape future après validation métier, DSI et sécurité.

### 1.5 Pertinence de l'IA

Une solution simple de règles métier ou de dashboard suffit pour expliquer certaines situations évidentes, par exemple un budget très tendu ou un vol trop coûteux. L'IA devient pertinente pour tester si la combinaison de plusieurs variables permet d'anticiper la satisfaction client. Les résultats montrent toutefois que le modèle pré-voyage reste limité : il doit donc être utilisé comme signal d'aide à l'analyse, et non comme décision automatique.


### 1.6 Datasheet structurée du jeu de données

Cette sous-section documente le dataset de référence utilisé dans ce notebook : `data/versions/v2_2_signal_enrichment/travel_planning_dataset_v2_2_minimal.csv`.

#### Motivation et contexte

Le dataset initial fourni est synthétique et anonymisé. La version rattrapage construit une version enrichie `v2.2-minimal` afin de tester une hypothèse métier : un dataset plus cohérent et plus porteur de signal pré-voyage permet-il d'obtenir un modèle plus exploitable ?

#### Composition

| Élément | Valeur retenue |
| --- | --- |
| Source initiale | `data/Examen_travel_planning_dataset.csv` |
| Dataset de référence notebook | `data/versions/v2_2_signal_enrichment/travel_planning_dataset_v2_2_minimal.csv` |
| Nombre de lignes | `3000` |
| Nombre de colonnes | `24` |
| Version source nettoyée | `1378` lignes |
| Lignes synthétiques ajoutées | `1622` lignes |
| Cible principale | satisfaction haute (`4-5`) vs satisfaction faible/moyenne (`1-3`) |

#### Usages recommandés

- Tester un prototype de prédiction pré-voyage.
- Comparer des modèles simples et explicables.
- Démontrer une chaîne complète : données, entraînement, API, monitoring et CI/CD.

#### Limitations

- Une partie du dataset `v2.2` est synthétique : les performances doivent être confirmées sur données réelles.
- Le modèle ne doit pas servir à refuser une offre, modifier un prix ou automatiser une décision client.
- Le score reste une aide à l'analyse, avec supervision humaine obligatoire.

#### Distribution et maintenance

Le dataset est versionné dans `data/versions/`. Toute nouvelle version doit documenter la source, les transformations, les colonnes ajoutées, les métriques obtenues et la décision de promotion ou de rejet.


## Partie 2 - Risques éthiques, sociétaux et registre RGPD

Dans ce notebook, le traitement porte sur le dataset enrichi `v2.2-minimal`, dérivé du fichier brut `data/Examen_travel_planning_dataset.csv`. Ce fichier est un dataset synthétique et anonymisé. Il ne contient pas de nom, prénom, email, téléphone, adresse, numéro de passeport ou identifiant client réel. Le registre ci-dessous documente donc le traitement réalisé dans ce projet, et non un traitement opérationnel de données clients réelles.

#### Finalités du traitement

| Finalité | Description |
| --- | --- |
| Documentation du dataset | Décrire le jeu de données utilisé pour répondre au besoin métier de planification de voyages. |
| Analyse de cohérence métier | Vérifier la qualité du fichier : valeurs manquantes, incohérences, doublons, valeurs atypiques. |
| Modélisation pré-voyage | Tester la capacité des variables disponibles avant le séjour à expliquer la satisfaction client. |
| Modélisation post-voyage | Tester l'apport des variables observées pendant ou après le séjour : `imprevus`, `respect_budget`, `reorganisation_necessaire`. |

#### Catégories de données collectées

| Catégorie | Colonnes concernées | Statut dans ce projet |
| --- | --- | --- |
| Identifiant technique | `trip_id` | Identifiant synthétique de ligne, non rattaché à une personne réelle. |
| Profil voyageur simulé | `client_type` | Catégorie générique : famille, couple, solo, business, senior. |
| Caractéristiques du séjour | `destination`, `saison`, `duree_jours`, `type_hebergement`, `activite_principale`, `meteo_prevue` | Variables descriptives du voyage fictif. |
| Données budgétaires simulées | `budget_total`, `prix_vol` | Montants fictifs utilisés pour l'analyse et la modélisation. |
| Événements post-voyage simulés | `imprevus`, `reorganisation_necessaire`, `respect_budget` | Variables opérationnelles fictives connues après ou pendant le séjour. |
| Satisfaction et avis fictifs | `satisfaction_client`, `retour_client` | Score et commentaire synthétiques, sans auteur identifiable. |

Aucune donnée personnelle directement identifiable n'est traitée dans le périmètre actuel du notebook.

#### Durée de conservation

| Élément conservé | Durée retenue pour ce projet |
| --- | --- |
| Dataset brut synthétique | Conservé dans `data/` pendant la durée du projet. |
| Notebooks d'analyse | Conservés dans `notebooks/` pour assurer la traçabilité des choix et des résultats. |
| Documentation projet | Conservée dans `docs/` et dans le notebook pour justifier la démarche. |

Comme le dataset est synthétique, il n'y a pas de durée de conservation liée à des personnes concernées identifiables dans ce projet.


### 2.1 Analyse des risques éthiques et sociétaux

Cette section synthétise les risques éthiques, sociétaux, juridiques et environnementaux propres à TravelMind. Le détail de conformité AI Act est traité une seule fois dans la section 2.2.

#### Positionnement éthique du projet

| Point de cadrage | Position retenue |
| --- | --- |
| Nature de la solution | Outil d'aide à l'analyse et à la recommandation, pas une décision automatique |
| Domaine | Planification de voyages haut de gamme et anticipation de la satisfaction client |
| Données | Dataset synthétique et anonymisé, sans données personnelles réelles |
| Sortie du modèle | Prédiction probabiliste en 3 classes : insatisfait, neutre, satisfait |
| Usage autorisé | Aide à la personnalisation, priorisation des contrôles humains, analyse qualité |

#### Risques identifiés et mesures retenues

| Risque | Conséquence possible | Mesure appliquée dans le projet |
| --- | --- | --- |
| Surinterprétation du score | Le client ou le conseiller peut croire que la prédiction est certaine | Affichage des probabilités, du niveau de confiance, des limites du modèle et maintien d'une supervision humaine |
| Performance pré-voyage modeste | Le score peut être utilisé comme une certitude alors que le signal pré-voyage est faible | Positionnement du modèle comme aide indicative, lecture des métriques, matrice de confusion et revue humaine obligatoire |
| Fuite de données | Utiliser des informations post-voyage pour prédire avant départ fausserait les performances | Séparation stricte pré-voyage / post-voyage et exclusion de `imprevus`, `respect_budget`, `reorganisation_necessaire` et `retour_client` du modèle pré-voyage |
| Automatisation excessive | Le score pourrait déclencher une décision commerciale sans validation humaine | Usage interdit pour refus automatique, tarification individualisée injustifiée ou remplacement du conseiller |
| Données synthétiques | Les résultats peuvent ne pas refléter le comportement de clients réels | Documentation des limites, validation métier requise et réévaluation avant usage avec données réelles |
| Données personnelles futures | L'ajout de données clients réelles pourrait créer des risques RGPD | Registre de traitement, minimisation, anonymisation/pseudonymisation, politique d'accès et validation juridique avant diffusion |
| Manque d'alphabétisation IA | Les utilisateurs internes peuvent mal interpréter les sorties du modèle | Formation courte, guide utilisateur, consignes d'usage, sensibilisation aux biais et procédure de revue humaine |
| Dérive des données en exploitation | Les profils de voyages saisis peuvent s'éloigner du dataset d'entraînement | Monitoring `/monitoring/drift`, alertes, journalisation JSONL et stratégie de réentraînement |
| Impact environnemental | Entraînements lourds, NLP avancé ou optimisations massives peuvent augmenter l'empreinte carbone | Modèles tabulaires sobres, CodeCarbon, NLP non retenu dans le modèle industrialisé et expériences lourdes isolées |

#### Biais potentiels à surveiller

| Axe de biais | Pourquoi le surveiller | Contrôle prévu dans TravelMind |
| --- | --- | --- |
| Budget, prix du vol et hébergement | Ces variables peuvent agir comme proxys socio-économiques et influencer injustement la recommandation | Analyse des erreurs par tranches de budget, contrôle des variables importantes, interdiction d'usage pour exclure ou défavoriser un client |
| Destination, saison et météo prévue | Certaines destinations ou périodes peuvent dominer le dataset et réduire la qualité sur les cas rares | Contrôle de distribution, suivi de dérive via `/monitoring/drift`, comparaison des performances par destination si volume suffisant |
| Classe de satisfaction | Les classes ne sont pas parfaitement équilibrées, ce qui peut favoriser la classe majoritaire | Suivi de `macro_f1`, `balanced_accuracy`, matrice de confusion, baseline `DummyClassifier` et validation croisée |
| Données synthétiques | Le dataset peut ne pas représenter toute la diversité de clients réels d'une agence haut de gamme | Documentation des limites, validation métier avant production, test sur nouvelles données réelles anonymisées si disponibles |
| Texte libre `retour_client` | Le sentiment exprimé peut être très proche de la satisfaction finale et créer une fuite de signal | NLP conservé comme exploration qualitative, non retenu dans le modèle pré-voyage industrialisé |
| Dérive en exploitation | Les voyages saisis dans l'API peuvent évoluer par rapport au jeu d'entraînement | Logs JSONL, dashboard KPI, alertes de dérive, seuils de monitoring et stratégie de réentraînement |

#### Acteurs à informer

| Acteur | Information à transmettre | Action attendue |
| --- | --- | --- |
| Commanditaire / direction métier | Objectif du modèle, performances modestes du pré-voyage, limites du dataset synthétique et usage non automatique | Valider le positionnement comme aide à la décision, pas comme outil de décision automatique |
| Conseillers voyage | Interprétation des probabilités, niveau de confiance, limites du modèle, cas nécessitant une revue humaine | Utiliser TravelMind comme support de conseil et garder la responsabilité de la recommandation finale |
| Clients finaux | Information claire qu'une IA peut assister la personnalisation du séjour et que la proposition peut être revue par un conseiller | Garantir transparence, compréhension et possibilité d'explication humaine |
| Référent juridique / RGPD / DPO | Nature synthétique du dataset actuel, conditions d'ajout de données réelles, registre de traitement, durée de conservation et droits des personnes | Valider le cadre légal avant toute utilisation de données clients réelles ou diffusion externe |
| Équipe data / technique | Variables autorisées, variables exclues, pipeline sans fuite, monitoring, dérive, quality gate et réentraînement | Maintenir le modèle, contrôler les performances et documenter chaque évolution |
| DSI / sécurité | Architecture locale, API, logs, Docker, accès aux données et conditions d'exposition éventuelle hors poste local | Définir les règles d'hébergement, d'accès, de sauvegarde et de sécurité avant production |


### 2.2 Conformité à la loi européenne sur l'IA (AI Act)

Cette section documente les réponses au questionnaire AI Act pour TravelMind, en tenant compté d'un usage destiné aux clients finaux de l'agence de voyage. 

#### Références réglementaires utilisées

Sources officielles : https://artificialintelligenceact.eu/fr/evaluation/verificateur-de-conformite-a-l-acte-de-l-ai-de-l-ue/

#### Positionnement AI Act retenu

Le tableau ci-dessous reprend les 9 questions du questionnaire AI Act applicables au positionnement de TravelMind. Les réponses sont documentées pour le cas du projet : solution de recommandation et d'anticipation de satisfaction pour une agence de voyages, destinée à des utilisateurs finaux, sans décision automatique.

| Elément du questionnaire | Réponse | Justification pour TravelMind |
| --- | --- | --- |
| Type d'entité | Fournisseur/ déployeur | Le projet développe le modèle, l'API et le dashboard. |
| Modification par un acteur aval | Aucune de ces réponses | Aucun déployeur, distributeur ou importateur aval n'a changé la marque, la finalité ou le fonctionnement substantiel du système. |
| Catégories haut risque produits | Aucune | TravelMind n'est pas un composant de sûreté pour aviation civile, véhicules, systèmes ferroviaires, équipements marins ou véhicules agricoles. |
| Systèmes interdits | Aucune | TravelMind ne manipule pas les personnes, n'exploite pas leurs vulnérabilités, ne fait pas de biométrie, notation sociale, police prédictive, reconnaissance faciale ou reconnaissance des émotions. |
| Systèmes transparents | Aucune dans le cadre actuel | TravelMind ne génère pas de contenu synthétique audio, image, vidéo ou texte publié pour informer le public ; ne fait pas de reconnaissance des émotions, de catégorisation biométrique ou de deepfake. |
| Classement haut risque global | non haut risque dans le cadre actuel | Le cas d'usage concerne la personnalisation de voyages et l'aide à l'anticipation de satisfaction. Il ne conditionne pas l'accès à un service essentiel, à l'emploi, au crédit, à l'éducation, à la justice ou à un service public. À réévaluer si la finalité change. |
| Champ d'application | Je suis établi ou situé dans l'UE ; les résultats de mon système d'IA sont utilisés dans l'UE | Le projet est conçu pour une agence de voyage opérant dans l'UE et les recommandations TravelMind sont destinées à des clients ou conseillers situés dans l'UE. |
| Systèmes exclus | Aucune | TravelMind n'est pas développé exclusivement à des fins militaires, n'est pas utilisé par des autorités de pays tiers pour l'application de la loi, n'est pas une simple activité de R&D isolée, n'est pas un composant libre indépendant et n'est pas un usage personnel non professionnel. |

#### Résultat : Obligation d'alphabétisation IA

Voici la démarche d'alphabétisation IA prévue pour TravelMind :

| Mesure | Application pour TravelMind |
| --- | --- |
| Formation courte | Sensibilisation pour conseillers, métier, technique et monitoring |
| Guide utilisateur | Explication des entrees, sorties, classes, probabilites, niveau de confiance et limites |
| Règles d'usage | Interdiction d'utilisér TravelMind seul pour refuser une offre ou remplacer le conseiller |
| Supervision humaine | Le conseiller peut valider, corriger ou ignorer la recommandation |
| Gestion des alertes | Faible confiance, derive ou résultat incoherent = revue humaine obligatoire |
| Mise à jour périodique | Nouvelle sensibilisation après changement majeur du modèle, des données ou du cas d'usage |

#### Conclusion

TravelMind peut être utilisé comme support d'analyse et de personnalisation à condition de conserver une supervision humaine, de présenter les prédictions comme probabilistes, de surveiller les biais, d'informer les acteurs concernés et de ne pas utiliser le score pour prendre une décision automatique individuelle.


## Partie 3 - Recentrage rattrapage et finalité

### Problème métier

Une agence de voyages haut de gamme souhaite identifier avant le départ les séjours susceptibles de générer une satisfaction faible ou moyenne, afin de renforcer l'accompagnement humain.

### Plus-value attendue

- Donner au conseiller un score indicatif de satisfaction probable.
- Prioriser les dossiers nécessitant une revue humaine.
- Structurer un pipeline reproductible pour préparer une future amélioration avec données réelles.

### Limite assumée

La prédiction ne remplace pas le conseiller. Avec les données actuelles, elle sert principalement de prototype analytique et industrialisable.


## Partie 4 - Imports et configuration


In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from app.modeling import (
    RANDOM_STATE,
    TARGET_COLUMN,
    add_base_features,
    clean_dataset,
    prepare_training_dataset,
    train_and_select_model,
)

V22_DIR = PROJECT_ROOT / "data" / "versions" / "v2_2_signal_enrichment"
V22_MINIMAL_DATA_PATH = V22_DIR / "travel_planning_dataset_v2_2_minimal.csv"
DATA_PATH = V22_MINIMAL_DATA_PATH
MODEL_PATH = PROJECT_ROOT / "models" / "model_pre_voyage.pkl"
METADATA_PATH = PROJECT_ROOT / "models" / "model_pre_voyage_metadata.json"

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 30)
sns.set_theme(style="whitegrid")


## Partie 5 - Construction et chargement du dataset `v2.2`

Le dataset utilisé pour la version rattrapage est construit avant l'entraînement afin de clarifier la chaîne de données : source brute, enrichissement, version minimale retenue, puis modélisation.


### 5.1 Création du dataset `v2.2` à 3000 lignes

Cette étape centralise la construction du dataset enrichi utilisé pour le rattrapage. Elle reprend le script `scripts/build_signal_enrichment_dataset.py` sans recopier deux fois le même code dans le notebook.

La version `v2.2` est générée à partir des données nettoyées et enrichies, puis complétée jusqu'à `3000` lignes par perturbation contrôlée. L'objectif est de tester une hypothèse : un dataset avec plus de signal métier permet-il d'obtenir un modèle plus exploitable ?

La cellule ci-dessous ne regénère le dataset que si les fichiers `v2.2` sont absents, ce qui évite les doublons et rend l'exécution plus rapide.


In [ ]:
# Construction ou verification du dataset enrichi v2.2.
# Le code metier complet est versionne dans scripts/build_signal_enrichment_dataset.py.
# Ici, le notebook appelle ce script pour garder une seule source de verite.
from scripts.build_signal_enrichment_dataset import main as build_signal_enrichment_dataset

FORCE_REBUILD_DATASET_V22 = False

V22_REPORT_PATHS = {
    "v22_full": V22_DIR / "signal_enrichment_report.json",
    "v22_light": V22_DIR / "light_features_experiment_report.json",
    "v22_minimal": V22_DIR / "minimal_features_experiment_report.json",
    "v22_minimal_balanced": V22_DIR / "minimal_balanced_classification_report.json",
    "v22_hyperopt": V22_DIR / "hyperparameter_optimization_report.json",
    "v22_overfitting": V22_DIR / "overfitting_diagnostic_report.json",
}

fichiers_v22_attendus = [V22_MINIMAL_DATA_PATH, *V22_REPORT_PATHS.values()]
fichiers_v22_absents = [path for path in fichiers_v22_attendus if not path.exists()]

if FORCE_REBUILD_DATASET_V22 or fichiers_v22_absents:
    print("Generation du dataset v2.2 et des rapports associes...")
    build_signal_enrichment_dataset()
else:
    print("Dataset v2.2 deja disponible : regeneration non necessaire.")


def charger_json(path: Path) -> dict:
    with Path(path).open("r", encoding="utf-8") as fichier:
        return json.load(fichier)

rapports_rattrapage = {
    nom: charger_json(path)
    for nom, path in V22_REPORT_PATHS.items()
}

rapport_v22 = rapports_rattrapage["v22_full"]
rapport_light = rapports_rattrapage["v22_light"]
rapport_minimal = rapports_rattrapage["v22_minimal"]

resume_datasets_v22 = pd.DataFrame([
    {
        "version": "v2.2_complete",
        "lignes": rapport_v22["rows"],
        "colonnes": rapport_v22["columns_after"],
        "lignes_source": rapport_v22["original_rows"],
        "lignes_synthetiques": rapport_v22["synthetic_rows_added"],
    },
    {
        "version": "v2.2_light",
        "lignes": rapport_light["rows"],
        "colonnes": rapport_light["columns"],
        "lignes_source": rapport_v22["original_rows"],
        "lignes_synthetiques": rapport_v22["synthetic_rows_added"],
    },
    {
        "version": "v2.2_minimal",
        "lignes": rapport_minimal["rows"],
        "colonnes": rapport_minimal["columns"],
        "lignes_source": rapport_v22["original_rows"],
        "lignes_synthetiques": rapport_v22["synthetic_rows_added"],
    },
])

DATA_PATH = V22_MINIMAL_DATA_PATH
if not DATA_PATH.exists():
    raise FileNotFoundError("Le dataset v2.2-minimal doit etre cree avant de poursuivre le notebook.")

print(f"Dataset retenu pour la suite : {DATA_PATH.relative_to(PROJECT_ROOT)}")
display(resume_datasets_v22)


### 5.2 Dataset utilisé dans tout le notebook

Le notebook utilise exclusivement le fichier `data/versions/v2_2_signal_enrichment/travel_planning_dataset_v2_2_minimal.csv`, généré à l'étape précédente.

Cette version est retenue car elle limite le nombre de colonnes au strict utile, réduit le bruit et garde les variables les plus défendables métier. Si le fichier est absent, l'exécution s'arrête explicitement afin d'éviter de mélanger plusieurs versions de données dans le même notebook.


In [ ]:
df_raw = pd.read_csv(DATA_PATH)
print(f"Dataset utilise : {DATA_PATH.relative_to(PROJECT_ROOT)}")
print(f"Nombre de lignes : {df_raw.shape[0]}")
print(f"Nombre de colonnes : {df_raw.shape[1]}")
display(df_raw.head())


In [ ]:
datasheet_variables = pd.DataFrame([
    {"variable": "trip_id", "role": "identifiant", "usage": "Exclu du modèle"},
    {"variable": "client_type", "role": "profil client", "usage": "Entrée pré-voyage"},
    {"variable": "budget_total", "role": "budget global", "usage": "Entrée pré-voyage"},
    {"variable": "destination", "role": "destination", "usage": "Entrée pré-voyage"},
    {"variable": "saison", "role": "période", "usage": "Entrée pré-voyage"},
    {"variable": "duree_jours", "role": "durée", "usage": "Entrée pré-voyage"},
    {"variable": "type_hebergement", "role": "hébergement", "usage": "Entrée pré-voyage"},
    {"variable": "prix_vol", "role": "coût transport", "usage": "Entrée pré-voyage"},
    {"variable": "meteo_prevue", "role": "contexte météo", "usage": "Entrée pré-voyage"},
    {"variable": "activite_principale", "role": "centre d'intérêt", "usage": "Entrée pré-voyage"},
    {"variable": "satisfaction_client", "role": "cible", "usage": "Score à prédire"},
    {"variable": "imprevus", "role": "événement post-voyage", "usage": "Exclu du modèle pré-voyage"},
    {"variable": "reorganisation_necessaire", "role": "résultat opérationnel", "usage": "Exclu du modèle pré-voyage"},
    {"variable": "respect_budget", "role": "résultat budgétaire", "usage": "Exclu du modèle pré-voyage"},
    {"variable": "retour_client", "role": "avis post-voyage", "usage": "Exclu du modèle pré-voyage"},
])
display(datasheet_variables)


## Partie 6 - Préparation des données et feature engineering

### 6.1 Finalité

La préparation des données vise un objectif précis : construire un dataset **cohérent, traçable et exploitable pour un modèle pré-voyage**.

La plus-value n'est pas de multiplier les traitements, mais de sécuriser les données utilisées par le modèle :

- conserver uniquement les informations disponibles avant le départ ;
- supprimer les cas impossibles ou contradictoires pour éviter un apprentissage incohérent ;
- centraliser les règles métier dans `configs/business_rules.json` ;
- laisser les traitements apprenants dans le pipeline `scikit-learn` après le split train/test.

### 6.2 Principe de décision

Seules les incohérences critiques sont supprimées. Les cas discutables mais métier possibles sont conservés et documentés, afin de ne pas appauvrir artificiellement le dataset.


In [ ]:
from app.config import load_business_rules

# Lecture des règles métier centralisées.
# Elles évitent de modifier le notebook à chaque évolution de borne, catégorie ou seuil.
business_rules = load_business_rules()

contraintes_numeriques = pd.DataFrame([
    {
        "variable": variable,
        "minimum": limites.get("min"),
        "maximum": limites.get("max"),
        "objectif": {
            "duree_jours": "écarter les durées irréalistes",
            "budget_total": "écarter les budgets hors périmètre métier",
            "prix_vol": "écarter les coûts de vol hors périmètre métier",
        }.get(variable, "contrôle métier"),
    }
    for variable, limites in business_rules["api_constraints"].items()
])

categories_fermees = pd.DataFrame([
    {
        "variable": variable,
        "valeurs_autorisees": ", ".join(valeurs),
        "objectif": "éviter les catégories inconnues sur les champs métier fermés",
    }
    for variable, valeurs in business_rules["allowed_categories"].items()
])

display(contraintes_numeriques)
display(categories_fermees)


In [ ]:
# Contrôles d'intégrité initiaux.
# Ces contrôles servent à distinguer les anomalies critiques des simples cas métier atypiques.
imprevus_norm = (
    df_raw["imprevus"]
    .fillna("aucun")
    .astype("string")
    .str.strip()
    .str.lower()
    .replace({"": "aucun", "nan": "aucun"})
)

def compter_hors_bornes(variable):
    limites = business_rules["api_constraints"][variable]
    valeurs = pd.to_numeric(df_raw[variable], errors="coerce")
    masque = pd.Series(False, index=df_raw.index)
    if limites.get("min") is not None:
        masque |= valeurs < limites["min"]
    if limites.get("max") is not None:
        masque |= valeurs > limites["max"]
    return masque

masque_hors_bornes = pd.Series(False, index=df_raw.index)
for variable in business_rules["api_constraints"]:
    masque_hors_bornes |= compter_hors_bornes(variable)

controle_qualite_initial = pd.DataFrame([
    {
        "controle": "doublons trip_id",
        "nb_lignes": int(df_raw["trip_id"].duplicated().sum()),
        "decision": "supprimer si présent",
        "plus_value": "éviter le double comptage d'un séjour",
    },
    {
        "controle": "satisfaction absente ou hors échelle 1-5",
        "nb_lignes": int((df_raw[TARGET_COLUMN].isna() | ~df_raw[TARGET_COLUMN].between(1, 5)).sum()),
        "decision": "supprimer",
        "plus_value": "garantir une cible d'apprentissage valide",
    },
    {
        "controle": "valeurs hors bornes métier configurées",
        "nb_lignes": int(masque_hors_bornes.sum()),
        "decision": "supprimer",
        "plus_value": "écarter les séjours hors périmètre du cas d'usage",
    },
    {
        "controle": "prix_vol > budget_total",
        "nb_lignes": int((df_raw["prix_vol"] > df_raw["budget_total"]).sum()),
        "decision": "supprimer",
        "plus_value": "éviter une incohérence budgétaire forte",
    },
    {
        "controle": "aucun imprévu mais réorganisation nécessaire",
        "nb_lignes": int(((imprevus_norm == "aucun") & (df_raw["reorganisation_necessaire"] == 1)).sum()),
        "decision": "supprimer",
        "plus_value": "éviter une contradiction opérationnelle forte",
    },
])

display(controle_qualite_initial)

# Application du nettoyage stabilisé utilisé aussi par train.py.
df_clean, cleaning_report = clean_dataset(df_raw)
df_model = add_base_features(df_clean)

finalites_preparation = {
    "dataset_brut": "point de départ fourni",
    "cible_satisfaction_client_valide": "cible exploitable pour la régression",
    "contraintes_metier_config": "périmètre métier cohérent avec l'API",
    "coherence_initiale_prix_vol_budget_total": "budget total compatible avec le coût du vol",
    "reorganisation_sans_imprevu_declare": "cohérence opérationnelle minimale",
}

rapport_nettoyage = pd.DataFrame(cleaning_report)
rapport_nettoyage["lignes_supprimees_cumulees"] = len(df_raw) - rapport_nettoyage["nb_lignes"]
rapport_nettoyage["lignes_restantes_pct"] = (rapport_nettoyage["nb_lignes"] / len(df_raw) * 100).round(2)
rapport_nettoyage["finalite"] = rapport_nettoyage["etape"].map(finalites_preparation)

display(rapport_nettoyage)

print(f"Lignes initiales : {len(df_raw)}")
print(f"Lignes après nettoyage : {len(df_clean)}")
print(f"Lignes supprimées : {len(df_raw) - len(df_clean)}")


### 6.3 Traitements placés dans le pipeline

Les traitements qui apprennent des paramètres ne sont pas appliqués directement sur tout le dataset avant la séparation train/test.

Ils sont intégrés au pipeline `scikit-learn` :

- imputation des valeurs manquantes ;
- traitement IQR des valeurs aberrantes numériques ;
- standardisation des variables numériques continues ;
- encodage `OneHotEncoder` des variables catégorielles.

Cette organisation limite le risque de fuite de données entre le train et le test.

### 4.4 Conclusion C3 - plus-value de la préparation

La préparation réduit le dataset de `1500` à `1378` lignes, uniquement après suppression de cas invalides, hors périmètre ou fortement incohérents. Le dataset final est plus fiable, mieux aligné avec l'usage pré-voyage et réutilisable par le notebook, `train.py`, l'API et le monitoring.

La section C3 est donc recentée sur la finalité : **produire une base propre et défendable**, sans prêtendre améliorer artificiellement la performance du modèle.


### 6.4 Feature engineering retenu

Les nouvelles variables sont limitées à des informations disponibles avant le départ.


In [ ]:
x_pre, y_pre, cleaning_report = prepare_training_dataset(df_raw)

features_resume = pd.DataFrame([
    {"feature": "budget_par_jour", "definition": "budget_total / duree_jours", "raison": "Comparer les séjours de durées différentes"},
    {"feature": "part_vol_budget", "definition": "prix_vol / budget_total", "raison": "Mesurer le poids du transport dans le budget"},
    {"feature": "sejour_long", "definition": "1 si duree_jours >= 14", "raison": "Identifier les séjours longs"},
    {"feature": "meteo_risque", "definition": "1 si pluie ou variable", "raison": "Repérer un contexte météo moins favorable"},
    {"feature": "client_business", "definition": "1 si client_type = business", "raison": "Isoler les voyages professionnels"},
    {"feature": "hebergement_luxe", "definition": "1 si resort ou villa", "raison": "Identifier un niveau d'hébergement premium"},
])

display(features_resume)
print(f"Nombre total de variables d'entrée : {x_pre.shape[1]}")
display(pd.DataFrame({"features_modelisation": x_pre.columns}))


### 6.5 Variables exclues pour éviter la fuite de données

Les variables post-voyage sont volontairement exclues du modèle pré-voyage : elles ne sont pas connues au moment où l'agence veut anticiper la satisfaction.


In [ ]:
variables_exclues = pd.DataFrame([
    {"variable": "imprevus", "raison": "Connu pendant ou après le séjour"},
    {"variable": "reorganisation_necessaire", "raison": "Conséquence opérationnelle post-voyage"},
    {"variable": "respect_budget", "raison": "Résultat constaté après séjour"},
    {"variable": "retour_client", "raison": "Avis client post-voyage, trop proche de la cible"},
    {"variable": "trip_id", "raison": "Identifiant sans valeur prédictive métier"},
])

display(variables_exclues)


## Partie 7 - Modélisation finale rattrapage : classification binaire

La création du dataset `v2.2` est désormais documentée dans la partie préparation des données. Cette section conserve uniquement les résultats d'expériences et la décision finale.


### 7.1 Comparaison des versions du dataset

Trois variantes ont été testées :

- `v2.2_complete` : version complète avec toutes les colonnes d'enrichissement et de traçabilité ;
- `v2.2_light` : suppression des colonnes techniques, de traçabilité et de certaines variables intermédiaires ;
- `v2.2_minimal` : conservation des colonnes indispensables au pipeline et de quelques scores métier synthétiques.

La version `minimal` est plus lisible pour le jury et plus facile à défendre métier, tout en conservant des performances très proches de la version `light`.


In [ ]:
comparaison_versions_v22 = pd.DataFrame([
    {
        "version": "v2.2_complete",
        "meilleur_modele_regression": rapport_v22["comparison"]["best_regression_after"]["modele"],
        "MAE": rapport_v22["comparison"]["best_regression_after"]["mae"],
        "R2": rapport_v22["comparison"]["best_regression_after"]["r2"],
        "meilleur_modele_binaire": rapport_v22["comparison"]["best_binary_after"]["modele"],
        "F1_binaire": rapport_v22["comparison"]["best_binary_after"]["f1_1"],
        "ROC_AUC": rapport_v22["comparison"]["best_binary_after"]["roc_auc"],
    },
    {
        "version": "v2.2_light",
        "meilleur_modele_regression": rapport_light["best_regression_light"]["modele"],
        "MAE": rapport_light["best_regression_light"]["mae"],
        "R2": rapport_light["best_regression_light"]["r2"],
        "meilleur_modele_binaire": rapport_light["best_binary_light"]["modele"],
        "F1_binaire": rapport_light["best_binary_light"]["f1_1"],
        "ROC_AUC": rapport_light["best_binary_light"]["roc_auc"],
    },
    {
        "version": "v2.2_minimal",
        "meilleur_modele_regression": rapport_minimal["best_regression_minimal"]["modele"],
        "MAE": rapport_minimal["best_regression_minimal"]["mae"],
        "R2": rapport_minimal["best_regression_minimal"]["r2"],
        "meilleur_modele_binaire": rapport_minimal["best_binary_minimal"]["modele"],
        "F1_binaire": rapport_minimal["best_binary_minimal"]["f1_1"],
        "ROC_AUC": rapport_minimal["best_binary_minimal"]["roc_auc"],
    },
])

display(comparaison_versions_v22)


### 7.2 Distribution de la cible binaire dans `v2.2-minimal`

La version `v2.2-minimal` garde `3000` lignes et `24` colonnes.

La distribution exacte de la cible reste imparfaite mais acceptable : la classe positive binaire (`satisfaction 4-5`) représente environ un tiers des observations.

L'équilibrage retenu ne modifie pas le dataset : il est appliqué uniquement pendant l'entraînement avec `class_weight="balanced"`.


In [ ]:
dataset_minimal_path = V22_MINIMAL_DATA_PATH
df_v22_minimal = pd.read_csv(dataset_minimal_path)

distribution_5_classes = (
    df_v22_minimal["satisfaction_client"]
    .value_counts()
    .sort_index()
    .rename_axis("satisfaction_client")
    .reset_index(name="nombre")
)
distribution_5_classes["pourcentage"] = (
    distribution_5_classes["nombre"] / len(df_v22_minimal) * 100
).round(2)

display(distribution_5_classes)

y_binaire_v22 = (df_v22_minimal["satisfaction_client"] >= 4).astype(int)
distribution_binaire = (
    y_binaire_v22
    .value_counts()
    .sort_index()
    .rename_axis("classe_binaire")
    .reset_index(name="nombre")
)
distribution_binaire["libelle"] = distribution_binaire["classe_binaire"].map({
    0: "satisfaction 1-3",
    1: "satisfaction 4-5",
})
distribution_binaire["pourcentage"] = (
    distribution_binaire["nombre"] / len(df_v22_minimal) * 100
).round(2)

display(distribution_binaire[["classe_binaire", "libelle", "nombre", "pourcentage"]])


### 7.3 Modèles comparés avec équilibrage des classes

L'objectif final testé ici est :

- `0` : satisfaction faible ou moyenne (`1`, `2`, `3`) ;
- `1` : satisfaction haute (`4`, `5`).

La pondération des classes permet de compenser le déséquilibre sans créer de nouvelles lignes supplémentaires.

Le modèle retenu à ce stade est `LogisticRegression_balanced`, car il obtient le meilleur compromis sur `F1`, `recall` et `balanced accuracy`.


In [ ]:
rapport_balanced = rapports_rattrapage["v22_minimal_balanced"]
resultats_balanced = pd.DataFrame(rapport_balanced["results"])

display(resultats_balanced[[
    "modele",
    "class_weight",
    "accuracy",
    "balanced_accuracy",
    "precision_1",
    "recall_1",
    "f1_1",
    "roc_auc",
]])

print("Meilleur modèle équilibré :")
display(pd.DataFrame([rapport_balanced["best_model"]]))


### 7.4 Optimisation des hyperparamètres avec `RandomizedSearchCV`

Deux familles de modèles ont été optimisées :

- `LogisticRegression` : recherche sur `C`, `penalty`, `solver` ;
- `RandomForest` : recherche sur `n_estimators`, `max_depth`, `min_samples_leaf`, `min_samples_split`, `max_features`, `class_weight`.

La validation croisée est stratifiée en `5 folds` avec `F1` comme métrique d'optimisation, car la classe positive reste minoritaire.


In [ ]:
rapport_hyperopt = rapports_rattrapage["v22_hyperopt"]
resultats_hyperopt = pd.DataFrame(rapport_hyperopt["all_results_sorted"])
colonnes_hyperopt = [
    "modele",
    "accuracy",
    "balanced_accuracy",
    "precision_1",
    "recall_1",
    "f1_1",
    "roc_auc",
    "cv_best_f1",
]

display(resultats_hyperopt[[col for col in colonnes_hyperopt if col in resultats_hyperopt.columns]])

print("Gains vs meilleur modèle initial :")
display(pd.DataFrame([rapport_hyperopt["gains_vs_best_initial"]]))

print("Meilleurs hyperparamètres du modèle retenu :")
best_model_hyperopt = rapport_hyperopt["best_model_after_optimization"]
display(pd.DataFrame([{
    "modele": best_model_hyperopt["modele"],
    "best_params": best_model_hyperopt.get("best_params", {}),
}]))


### 7.5 Matrice de confusion du modèle optimisé

La matrice de confusion est exploitée ici pour lire concrètement les erreurs du modèle retenu sur le jeu de test. Elle est plus utile que l'affichage brut seul, car elle distingue les vrais satisfaits détectés, les satisfaits manqués et les fausses alertes.


In [ ]:
# Matrice de confusion du meilleur modèle optimisé.
# Classe 0 : satisfaction faible/moyenne (1-3)
# Classe 1 : satisfaction haute (4-5)
cm = best_model_hyperopt["confusion_matrix"]

matrice_confusion = pd.DataFrame(
    [
        [cm["tn"], cm["fp"]],
        [cm["fn"], cm["tp"]],
    ],
    index=["réel_1_3", "réel_4_5"],
    columns=["prédit_1_3", "prédit_4_5"],
)

total_test = sum(cm.values())
interpretation_cm = pd.DataFrame([
    {
        "indicateur": "vrais satisfaits détectés",
        "valeur": cm["tp"],
        "lecture": "clients réellement satisfaits et prédits satisfaits",
    },
    {
        "indicateur": "satisfaits manqués",
        "valeur": cm["fn"],
        "lecture": "clients satisfaits classés à tort en faible/moyen",
    },
    {
        "indicateur": "fausses alertes satisfaction haute",
        "valeur": cm["fp"],
        "lecture": "clients faibles/moyens prédits à tort satisfaits",
    },
    {
        "indicateur": "taille du test",
        "valeur": total_test,
        "lecture": "nombre total d'observations évaluées",
    },
])

display(matrice_confusion)
display(interpretation_cm)


### 7.6 Diagnostic overfitting du modèle optimisé

Le meilleur modèle après optimisation est `LogisticRegression_balanced_optimized`.

Le diagnostic compare :

- les performances sur le jeu d'entraînement ;
- les performances sur le jeu de test ;
- les écarts moyens en validation croisée.

L'objectif est de vérifier si le modèle mémorise le train ou s'il généralise correctement.


In [ ]:
rapport_overfitting = rapports_rattrapage["v22_overfitting"]

print("Métriques train/test :")
display(pd.DataFrame(rapport_overfitting["holdout_metrics"]))

print("Écarts train/test :")
display(pd.DataFrame([rapport_overfitting["gaps"]]))

print("Validation croisée - synthèse :")
cv_rows = []
for metrique, valeurs in rapport_overfitting["cross_validation_summary"].items():
    cv_rows.append({
        "metrique": metrique,
        "moyenne": valeurs["mean"],
        "ecart_type": valeurs["std"],
    })
display(pd.DataFrame(cv_rows))

print("Conclusion :")
print(rapport_overfitting["conclusion"])


### 7.7 Décision actualisée pour la version rattrapage

Au regard des dernières expériences, la meilleure piste n'est plus la régression sur la note exacte, mais la **classification binaire pré-voyage**.

Modèle candidat retenu : `LogisticRegression_balanced_optimized`.

Résultats clés :

- `F1 = 0.5995` ;
- `ROC AUC = 0.7320` ;
- `balanced accuracy = 0.7057` ;
- pas de signe fort d'overfitting (`écart F1 train/test = 0.0236`).

Interprétation métier : le modèle devient exploitable comme prototype pour estimer un risque de satisfaction haute ou non, avant le départ.

Réserve importante : la version `v2.2-minimal` contient des lignes synthétiques. Le modèle peut donc être industrialisé comme prototype contrôlé, mais il doit être validé sur des données réelles avant toute mise en production décisionnelle.


## Partie 8 - Architecture cible industrialisée

### 9.1 Besoin d'architecture

L'architecture doit permettre de sortir du notebook et de rendre le prototype testable dans un environnement technique cohérent.

Objectifs couverts :

- entraîner le modèle pré-voyage de façon reproductible ;
- exposer une prédiction via une API ;
- proposer une interface simple de test ;
- journaliser les prédictions ;
- surveiller les dérives ;
- valider automatiquement le projet avec la CI/CD ;
- conserver une architecture proportionnée à un prototype.

### 9.2 Contraintes prises en compte

- Performance modèle faible : le système ne doit pas prendre de décision automatique.
- Données synthétiques : la production nécessite une validation sur données réelles.
- Reproductibilité : l'entraînement doit pouvoir être rejoué hors notebook.
- Sobriété : modèle tabulaire simple, pas de NLP lourd en production.
- Portabilité : API conteneurisable avec Docker.
- Gouvernance : validation métier, DSI et juridique requise avant généralisation.

### 9.3 Architecture retenue

Le flux logique retenu est :

`Dataset brut` -> `train.py` -> `artefacts modèle` -> `API FastAPI` -> `Streamlit` -> `logs et monitoring` -> `CI/CD et Docker`.

Cette architecture est adaptée à un **prototype industrialisable**, pas à une production client autonome.


In [ ]:
architecture_briques = pd.DataFrame([
    {
        "ordre": 1,
        "brique": "Données",
        "fichiers": "data/versions/v2_2_signal_enrichment/travel_planning_dataset_v2_2_minimal.csv, configs/business_rules.json",
        "role": "source d'entraînement et règles métier centralisées",
        "contrainte_couverte": "traçabilité et cohérence métier",
        "statut": "mis en oeuvre",
    },
    {
        "ordre": 2,
        "brique": "Entraînement",
        "fichiers": "train.py, app/modeling.py",
        "role": "rejouer le nettoyage, le feature engineering et l'entraînement",
        "contrainte_couverte": "reproductibilité hors notebook",
        "statut": "mis en oeuvre",
    },
    {
        "ordre": 3,
        "brique": "Artefacts modèle",
        "fichiers": "models/model_pre_voyage.pkl, models/model_pre_voyage_metadata.json",
        "role": "stocker le pipeline entraîné, les métriques et le profil de référence",
        "contrainte_couverte": "traçabilité des performances",
        "statut": "mis en oeuvre",
    },
    {
        "ordre": 4,
        "brique": "API",
        "fichiers": "app/main.py, app/predictor.py, app/schemas.py",
        "role": "servir /health, /predict et les endpoints de monitoring",
        "contrainte_couverte": "intégration technique et validation des entrées",
        "statut": "mis en oeuvre",
    },
    {
        "ordre": 5,
        "brique": "Interface de test",
        "fichiers": "app_web.py",
        "role": "tester la prédiction sans requête API manuelle",
        "contrainte_couverte": "accessibilité pour le métier",
        "statut": "mis en oeuvre",
    },
    {
        "ordre": 6,
        "brique": "Monitoring",
        "fichiers": "app/monitoring.py, logs/predictions/predictions.jsonl",
        "role": "journaliser les prédictions, suivre les zones d'incertitude et le data drift",
        "contrainte_couverte": "surveillance et revue humaine",
        "statut": "mis en oeuvre",
    },
    {
        "ordre": 7,
        "brique": "CI/CD et conteneurisation",
        "fichiers": ".github/workflows/ci-cd.yml, Dockerfile, docker-compose.yml",
        "role": "tester, contrôler la quality gate et construire l'image Docker si nécessaire",
        "contrainte_couverte": "non-régression et portabilité",
        "statut": "mis en oeuvre sans déploiement serveur automatique",
    },
])

scenarios_architecture = pd.DataFrame([
    {
        "scenario": "Notebook seul",
        "avantage": "simple pour l'analyse",
        "limite": "pas exploitable par une application",
        "decision": "non retenu seul",
    },
    {
        "scenario": "Script + API locale",
        "avantage": "modèle rejouable et prédiction testable",
        "limite": "usage local uniquement",
        "decision": "retenu pour le prototype",
    },
    {
        "scenario": "Docker local",
        "avantage": "environnement portable",
        "limite": "ne suffit pas à une production distante",
        "decision": "retenu pour la portabilité",
    },
    {
        "scenario": "VPS ou cloud managé",
        "avantage": "accès distant et supervision avancée",
        "limite": "coût, sécurité et validation DSI nécessaires",
        "decision": "non retenu à ce stade",
    },
])

acteurs_a_consulter = pd.DataFrame([
    {
        "acteur": "Commanditaire métier",
        "point_a_valider": "usage autorisé du score et seuils d'action humaine",
        "statut": "à consulter avant généralisation",
    },
    {
        "acteur": "DSI",
        "point_a_valider": "hébergement, sécurité, logs, sauvegardes et disponibilité",
        "statut": "à consulter avant déploiement distant",
    },
    {
        "acteur": "DPO / juridique",
        "point_a_valider": "RGPD, AI Act, durée de conservation et information utilisateur",
        "statut": "à consulter avant données réelles",
    },
])

display(architecture_briques)
display(scenarios_architecture)
display(acteurs_a_consulter)


## Partie 9 - API, monitoring et endpoints

Endpoints disponibles :

- `GET /health` : vérifier que l'API fonctionne ;
- `POST /predict` : prédire le score de satisfaction pré-voyage ;
- `GET /monitoring/summary` : résumer les prédictions journalisées ;
- `GET /monitoring/drift` : comparer les nouvelles entrées au profil d'entraînement ;
- `GET /monitoring/alerts` : synthétiser les alertes et proposer une action.

Le monitoring suit les entrées API, les scores prédits, les zones d'incertitude et les dérives. Il ne mesure pas encore la vraie performance en production, car cela nécessiterait des satisfactions réelles collectées après séjour.


## Partie 10 - Mesure de performance et impacts

La mesure de performance est recentrée sur le modèle final rattrapage : une classification binaire pré-voyage.

### 10.1 Indicateurs retenus

- `accuracy` : part globale de bonnes prédictions.
- `balanced_accuracy` : performance moyenne équilibrée entre les deux classes.
- `precision_1` : fiabilité des prédictions de satisfaction haute.
- `recall_1` : capacité à retrouver les séjours réellement satisfaits.
- `f1_1` : compromis entre précision et rappel sur la classe satisfaction haute.
- `ROC AUC` : capacité du modèle à séparer les deux classes sur l'ensemble des seuils.
- `écart train/test` : contrôle du risque d'overfitting.
- `empreinte carbone` : sobriété de l'entraînement lorsque CodeCarbon est disponible.


In [ ]:
# Synthèse C8 : transformer les métriques du modèle final en décision d'exploitation.
metrics_c8 = best_model_hyperopt
gains_c8 = rapport_hyperopt["gains_vs_best_initial"]

performance_c8 = pd.DataFrame([
    {
        "indicateur": "accuracy",
        "valeur": round(metrics_c8["accuracy"], 4),
        "lecture": "part globale de bonnes prédictions",
        "conclusion": "niveau correct pour un prototype",
    },
    {
        "indicateur": "balanced_accuracy",
        "valeur": round(metrics_c8["balanced_accuracy"], 4),
        "lecture": "performance équilibrée entre les classes 0 et 1",
        "conclusion": "meilleure lecture que l'accuracy seule",
    },
    {
        "indicateur": "recall_1",
        "valeur": round(metrics_c8["recall_1"], 4),
        "lecture": "part des séjours satisfaits correctement détectés",
        "conclusion": "indicateur clé pour anticiper les séjours à forte satisfaction",
    },
    {
        "indicateur": "f1_1",
        "valeur": round(metrics_c8["f1_1"], 4),
        "lecture": "compromis précision / rappel sur la classe satisfaction haute",
        "conclusion": "meilleur compromis obtenu après optimisation",
    },
    {
        "indicateur": "ROC AUC",
        "valeur": round(metrics_c8["roc_auc"], 4),
        "lecture": "capacité de séparation entre satisfaction haute et faible/moyenne",
        "conclusion": "signal exploitable, à confirmer sur données réelles",
    },
    {
        "indicateur": "gain F1 vs meilleur initial",
        "valeur": round(gains_c8["f1_gain"], 4),
        "lecture": "gain apporté par l'optimisation des hyperparamètres",
        "conclusion": "gain faible mais positif",
    },
])

decision_exploitation = pd.DataFrame([
    {
        "usage": "Démonstrateur technique",
        "decision": "autorisé",
        "raison": "le pipeline, les métriques, l'API, le monitoring et la CI/CD sont testables",
    },
    {
        "usage": "Support d'analyse interne",
        "decision": "autorisé avec prudence",
        "raison": "le score aide à prioriser une revue humaine, sans automatiser la décision",
    },
    {
        "usage": "Aide à la décision commerciale autonome",
        "decision": "non autorisé en l'état",
        "raison": "validation nécessaire sur données réelles avant usage opérationnel",
    },
    {
        "usage": "Décision automatique client",
        "decision": "interdit",
        "raison": "risque éthique et juridique trop élevé",
    },
])

display(performance_c8)
display(decision_exploitation)


In [ ]:
# Bilan carbone : lecture de la dernière mesure CodeCarbon si elle existe.
# Si le fichier n'existe pas sur un autre poste, la cellule reste non bloquante.
codecarbon_path = PROJECT_ROOT / "logs" / "codecarbon" / "emissions_notebook_final.csv"

if codecarbon_path.exists():
    emissions_df = pd.read_csv(codecarbon_path)
    derniere_mesure_carbone = emissions_df.tail(1).copy()
    derniere_mesure_carbone["emissions_g_co2e"] = derniere_mesure_carbone["emissions"] * 1000
    colonnes_carbone = [
        "timestamp",
        "duration",
        "emissions",
        "emissions_g_co2e",
        "energy_consumed",
        "cpu_power",
        "ram_power",
        "country_iso_code",
    ]
    display(derniere_mesure_carbone[colonnes_carbone].round(8))
else:
    print("Mesure CodeCarbon absente sur ce poste. Lancer une mesure carbone si besoin.")


### 10.2 Conclusion C8 - décision d'exploitation

Les résultats rattrapage montrent une amélioration nette par rapport aux premières versions exploratoires : `F1 = 0.5995`, `ROC AUC = 0.7320` et `balanced_accuracy = 0.7057`.

Conclusion opérationnelle :

- le modèle peut être présenté comme **prototype industrialisable** ;
- il peut aider à prioriser une revue humaine avant le départ ;
- il ne doit pas déclencher de décision automatique ;
- la validation sur données réelles reste obligatoire avant une mise en production métier.

L'impact carbone reste limité car le modèle final est tabulaire, simple et sans NLP lourd en production.


## Partie 11 - Amélioration continue

La priorité n'est pas d'ajouter des modèles plus complexes, mais d'améliorer les données disponibles avant le départ.

Données à collecter pour une version future :

- attentes explicites du client ;
- préférences détaillées ;
- contraintes personnelles ;
- historique client ;
- canal de réservation ;
- niveau d'accompagnement souhaité ;
- satisfaction réelle observée après séjour.

Le réentraînement devra être réalisé uniquement après validation métier, contrôle qualité et comparaison avec le modèle précédent.


## Partie 12 - Synthèse finale

TravelMind démontre une démarche IA complète : cadrage, préparation des données, modélisation, industrialisation, API, interface, monitoring, Docker et CI/CD.

La version rattrapage clarifie la convergence :

- la régression pré-voyage sur la note exacte `1 à 5` reste peu performante sur le dataset initial ;
- l'enrichissement réel météo / qualité de l'air améliore le contexte mais ne suffit pas ;
- une version expérimentale `v2.2` à `3000` lignes, partiellement synthétique, permet de tester l'hypothèse d'un signal métier plus structuré ;
- la formulation la plus exploitable devient la **classification binaire** : satisfaction haute (`4-5`) vs satisfaction faible/moyenne (`1-3`).

Le meilleur candidat actuel est `LogisticRegression_balanced_optimized` sur `v2.2-minimal_balanced`.

Conclusion : le modèle est **industrialisable comme prototype contrôlé**, mais il ne doit pas être présenté comme prêt pour une production réelle tant qu'il n'a pas été validé sur des données réelles collectées par l'agence.
